# MINT-TTS — Colab quickstart

**Minimal Inference Needed for Text-to-Speech**: the model learns how much computation each token actually needs.

This notebook: install → verify → inspect the text frontend → download LJSpeech → preprocess → train → watch the complexity heatmaps.

**Runtime → Change runtime type → GPU** before running anything.

## 1. Clone and install

espeak-ng comes from pip (`espeakng-loader`), so there is no `apt-get` step.

In [ ]:
REPO = 'https://github.com/MohammedAly22/MINT-TTS.git'

import os, pathlib
if not pathlib.Path('MINT-TTS').exists():
    !git clone -q $REPO MINT-TTS
os.chdir('MINT-TTS')
!pip install -q -r requirements.txt
!python -c "import nltk; nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('cmudict', quiet=True)"

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


In [ ]:
!python -m pytest -q

## 2. Which frontend should you train with?

Before touching any data, look at what each text frontend does to homographs.
A backend marked `same` committed to one pronunciation for both contexts —
half of those commitments are wrong, and the acoustic model cannot undo them.

In [ ]:
!python scripts/inspect_frontend.py

In [ ]:
# Check the normaliser on the hard cases too
from mint_tts.text.normalizer import TextNormalizer
n = TextNormalizer()
for t in ["Dr. Smith paid $1,250.75 on 3/15/2024.",
          "Call +1 (555) 123-4567 or email a.smith@mit.edu.",
          "15 Oak St., Apt. 4B; take Oak Dr. north.",
          "The FBI and NASA met at 3:30 pm on March 3rd, 1987."]:
    print(f'{t}\n  -> {n(t)}\n')

## 3. LJSpeech (2.6 GB, ~3 minutes)

In [ ]:
import pathlib
if not pathlib.Path('data/LJSpeech-1.1/metadata.csv').exists():
    !mkdir -p data
    !curl -sL -o /tmp/ljs.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
    !tar -xjf /tmp/ljs.tar.bz2 -C data/
!python scripts/prepare_dataset.py --dataset ljspeech --root data/LJSpeech-1.1

## 4. Preprocess

Mel, pitch and energy per utterance, plus cached phonemisation and the corpus
symbol table. ~10 minutes with 4 workers.

In [ ]:
!python scripts/preprocess.py --config configs/exp2_token.yaml --workers 4

import json
stats = json.load(open('data/preprocessed/ljspeech/stats.json'))
print(f"{stats['n_utterances']} utterances, {stats['total_hours']:.1f} h, "
      f"vocab {stats['vocab_size']} ({stats['input_type']})")

## 4b. Check figures work, and get a real vocoder

**Figures.** The default renderer is matplotlib, which draws straight into
TensorBoard with nothing external to install. (The interactive plotly path
needs kaleido *plus* a real Chrome install, which Colab does not ship — that
is why it silently produced no IMAGES tab before.)

**Vocoder.** The default is Griffin-Lim, which is why `audio_target_vocoded`
sounds rough even though the data is clean — that is the *vocoder's* ceiling,
not the model's. Compare it against `audio_target_original` (the untouched
file). Install HiFi-GAN before judging quality.

In [ ]:
# Figures use matplotlib by default: no browser, no kaleido, nothing to install.
import numpy as np
from mint_tts.utils import plotting
fig = plotting.plot_token_complexity(list('record'), np.linspace(0.2, 1.0, 6),
                                     'self-check', max_steps=8)
ok = plotting.to_image_array(fig) is not None
plotting.close(fig)
print('backend:', plotting.backend(), '| figure export ->',
      'OK, TensorBoard will have an IMAGES tab' if ok else 'BROKEN')

# Optional: interactive plotly figures (hover shows token/word/depth).
# Ideal with wandb. For TensorBoard it also needs Chrome:
#   !pip install -q 'plotly>=6.1.1' 'kaleido>=1.0' && plotly_get_chrome -y
#   then add:  log.figure_backend=plotly  to the training overrides

# HiFi-GAN (optional but recommended -- the default Griffin-Lim is why
# audio_target_vocoded sounds rough). See scripts/download_vocoder.py.
# !python scripts/download_vocoder.py --hf-repo <user>/<repo> --hf-file generator_v1
# then add:  vocoder.name=hifigan  to the training overrides


## 5. TensorBoard (start it *before* training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5b. Survive a disconnect

Colab reclaims the VM without warning and everything under `/content` goes
with it. Write checkpoints to Drive instead, and a run that dies at step 32k
resumes from step 30k rather than from zero.

In [ ]:
# Resume after a disconnect (needs the Drive cell above)
# !python scripts/train.py --config configs/exp2_token.yaml \n#     --override train.output_dir=$RUN_DIR \n#     --resume $RUN_DIR/exp2_token/checkpoints/best.pt


## 6. Train

**In the IMAGES tab, in this order:**
1. `train/0/alignment_hard` — must become a clean diagonal. If it does not, stop; every complexity number downstream is meaningless until it does.
2. `probe/01_homograph/*/word_complexity` — both `record` bars should grow taller than `the`.
3. `probe/compute_by_group` — `easy` and `long_easy` should sit below `homograph`.

**In SCALARS:**
- `probe/contrast` above zero and holding — that is the claim, as one number.
- `probe/length_corr` **not** near 1.0 — near 1.0 means the router only learned sentence length.
- `compute/combined_norm` falling after `loss.compute.warmup_steps` while `val/mcd` stays flat.

Nothing about compute means anything before step 8000: the penalty is in warmup until then, on purpose — the model learns to speak first.

Colab sessions expire. Checkpoints land in `runs/<run_name>/checkpoints/`, and `--resume` picks them back up.

In [ ]:
# Routing is held at FULL depth until loss.compute.warmup_steps, so the model
# learns to speak before any compute pressure is applied. Until then
# enc_depth sits at max and compute/lambda is 0 -- that is intended.
RUN_DIR = globals().get('RUN_DIR', 'runs')   # set above if Drive is mounted

!python scripts/train.py --config configs/exp2_token.yaml \n    --override train.batch_size=16 train.amp=true train.max_steps=200000 \n               train.output_dir=$RUN_DIR train.save_every=2000 \n               loss.compute.warmup_steps=8000 loss.compute.ramp_steps=4000 \n               log.probe_every=2000 log.figure_every=2000 log.log_audio=true


In [ ]:
# Resume after a disconnect (needs the Drive cell above)
# !python scripts/train.py --config configs/exp2_token.yaml \n#     --override train.output_dir=$RUN_DIR \n#     --resume $RUN_DIR/exp2_token/checkpoints/best.pt


## 7. Synthesise, and see where the compute went

In [ ]:
from IPython.display import Audio, display
from mint_tts.inference.synthesize import Synthesizer
from mint_tts.utils import plotting

syn = Synthesizer.from_checkpoint('runs/exp2_token/checkpoints/best.pt')

for text in ['Hello, how are you doing?',
             'The record is broken by the record broker.',
             'I went to the store and I bought some milk and some bread and some eggs.']:
    res = syn(text, quality=0.9)
    print(res.summary())
    if res.wav is not None:
        display(Audio(res.wav.cpu().numpy(), rate=res.sample_rate))
    # plotting.show(), not fig.show(): on a headless backend fig.show() is a
    # no-op, and it also closes the figure so a long loop cannot leak them.
    plotting.show(plotting.plot_word_complexity(
        res.encoded.words, res.word_complexity, title=text[:60]))
    plotting.show(plotting.plot_token_complexity(
        res.encoded.tokens, res.token_complexity, title=text[:60],
        max_steps=syn.model.encoder.max_steps,
        words=res.encoded.words, word_ids=res.encoded.word_ids))


## 8. The budget knob: C*(x, q)

Compute should rise monotonically with the requested quality.

In [ ]:
# NOTE: this only varies once routing is trained, i.e. after
# loss.compute.warmup_steps. Before that the router is bypassed and every
# budget returns the same compute.
text = 'The record is broken by the record broker.'
qs, cs = [], []
for q in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    r = syn(text, quality=q)
    qs.append(q); cs.append(float(r.token_complexity.mean()))
    print(f'q={q:<4} compute={cs[-1]:.3f}  FLOPs={r.flops.total:.2e}  '
          f'saving={r.flops.saving*100:5.1f}%  {r.latency_ms:6.1f} ms')
plotting.show(plotting.plot_compute_curve(
    cs, qs, title='requested quality vs allocated compute'))


## 9. Benchmark and compute curves

In [ ]:
!python scripts/benchmark.py --checkpoint runs/exp2_token/checkpoints/best.pt \
    --devices cpu cuda --budgets 0.1 0.5 0.9 1.0

!python scripts/compute_curve.py --checkpoint runs/exp2_token/checkpoints/best.pt \
    --index data/preprocessed/ljspeech/test.jsonl --limit 100

## 10. The comparison that matters

One adaptive run proves nothing on its own. Train the dense baselines, then build the matrix:

```bash
python scripts/train.py --config configs/exp0_dense.yaml
python scripts/train.py --config configs/exp0_dense_shallow.yaml

python scripts/evaluate.py \
    --checkpoints runs/exp0_dense/checkpoints/best.pt \
                  runs/exp0_dense_shallow/checkpoints/best.pt \
                  runs/exp2_token/checkpoints/best.pt \
    --index data/preprocessed/ljspeech/test.jsonl --asr wav2vec2 --benchmark
```

The result worth reporting is `exp2_token` matching `exp0_dense` on quality while sitting near `exp0_dense_shallow` on FLOPs and CPU RTF.

See `docs/EXPERIMENTS.md` for the decision rules — and `docs/HYPOTHESIS.md` for what would falsify the whole idea.